In [0]:
# ============================================================
# CELL 1: Imports & Read Silver Layer
# ============================================================

from pyspark.sql.functions import (
    col, sum as spark_sum, avg, round as spark_round,
    count, when, rank, desc, current_timestamp, lit,
    date_format
)
from pyspark.sql.window import Window

silver_path = "/Volumes/healthcare_pipeline/ae_data/silver_volume/delta/ae_attendances_clean"

silver_df = spark.read.format("delta").load(silver_path)

print(f"Records from Silver: {silver_df.count()}")
print(f"Columns: {len(silver_df.columns)}")

Records from Silver: 2370
Columns: 27


In [0]:
# ============================================================
# CELL 2: Build Consolidated Totals
# ============================================================

silver_df = silver_df.withColumn(
    "total_attendances",
    col("a_e_attendances_type_1") + col("a_e_attendances_type_2") +
    col("a_e_attendances_other_a_e_department")
).withColumn(
    "total_4hr_breaches",
    col("attendances_over_4hrs_type_1") + col("attendances_over_4hrs_type_2") +
    col("attendances_over_4hrs_other_department")
).withColumn(
    "total_emergency_admissions",
    col("emergency_admissions_via_a_e_type_1") + col("emergency_admissions_via_a_e_type_2") +
    col("emergency_admissions_via_a_e_other_a_e_department") + col("other_emergency_admissions")
)

print("Consolidated totals built")
silver_df.select("org_name", "total_attendances", "total_4hr_breaches", "total_emergency_admissions").show(5)

Consolidated totals built
+--------------------+-----------------+------------------+--------------------------+
|            org_name|total_attendances|total_4hr_breaches|total_emergency_admissions|
+--------------------+-----------------+------------------+--------------------------+
|CORNWALL PARTNERS...|                0|                 0|                        94|
|EAST BERKS PRIMAR...|                0|                 0|                         0|
|WORCESTERSHIRE AC...|            20338|              6898|                      3608|
|THE DUDLEY GROUP ...|            14228|              3003|                      6308|
|HULL UNIVERSITY T...|            13999|              6071|                      6058|
+--------------------+-----------------+------------------+--------------------------+
only showing top 5 rows


## **How is the NHS performing month over month?**

In [0]:
# ============================================================
# CELL 3: Gold Table 1 — Monthly Summary
# ============================================================

gold_monthly_summary = silver_df \
    .groupBy("period_date") \
    .agg(
        spark_sum("total_attendances").alias("total_attendances"),
        spark_sum("total_4hr_breaches").alias("total_4hr_breaches"),
        spark_sum("total_emergency_admissions").alias("total_emergency_admissions"),
        count("org_code").alias("hospitals_reporting")
    ) \
    .withColumn(
        "breach_rate_pct",
        spark_round((col("total_4hr_breaches") / col("total_attendances")) * 100, 2)
    ) \
    .withColumn(
        "target_met",
        when(col("breach_rate_pct") <= 5, "YES").otherwise("NO")
    ) \
    .withColumn("_gold_timestamp", current_timestamp()) \
    .orderBy("period_date")

print("Monthly Summary — Gold Table 1")
display(gold_monthly_summary)

Monthly Summary — Gold Table 1


period_date,total_attendances,total_4hr_breaches,total_emergency_admissions,hospitals_reporting,breach_rate_pct,target_met,_gold_timestamp
2025-06-01,2268166,566734,535941,201,24.99,NO,2026-06-30T12:57:48.598Z
2025-07-01,2320374,560285,559392,201,24.15,NO,2026-06-30T12:57:48.598Z
2025-08-01,2182723,539040,526399,201,24.7,NO,2026-06-30T12:57:48.598Z
2025-09-01,2227502,567723,535580,201,25.49,NO,2026-06-30T12:57:48.598Z
2025-10-01,2317742,610640,553499,197,26.35,NO,2026-06-30T12:57:48.598Z
2025-11-01,2263562,596066,531747,197,26.33,NO,2026-06-30T12:57:48.598Z
2025-12-01,2239924,599599,542188,197,26.77,NO,2026-06-30T12:57:48.598Z
2026-01-01,2235696,629727,546136,197,28.17,NO,2026-06-30T12:57:48.598Z
2026-02-01,2043572,541635,493015,197,26.5,NO,2026-06-30T12:57:48.598Z
2026-03-01,2351347,549428,553892,199,23.37,NO,2026-06-30T12:57:48.598Z


## **Which hospitals are struggling most, and which are performing best ?**

In [0]:
# ============================================================
# CELL 4: Gold Table 2 — Hospital Performance Ranking
# ============================================================

window_spec = Window.orderBy(desc("breach_rate_pct"))

gold_hospital_ranking = silver_df \
    .groupBy("org_code", "org_name", "parent_org") \
    .agg(
        spark_sum("total_attendances").alias("total_attendances"),
        spark_sum("total_4hr_breaches").alias("total_breaches"),
        spark_sum("total_emergency_admissions").alias("total_admissions")
    ) \
    .filter(col("total_attendances") > 0) \
    .withColumn(
        "breach_rate_pct",
        spark_round((col("total_breaches") / col("total_attendances")) * 100, 2)
    ) \
    .withColumn("performance_rank", rank().over(window_spec)) \
    .withColumn(
        "performance_band",
        when(col("breach_rate_pct") <= 5, "GREEN")
        .when(col("breach_rate_pct") <= 10, "AMBER")
        .otherwise("RED")
    ) \
    .withColumn("_gold_timestamp", current_timestamp())

print("Hospital Performance Ranking — Gold Table 2 (worst performers first)")
display(gold_hospital_ranking.orderBy("performance_rank").limit(15))

Hospital Performance Ranking — Gold Table 2 (worst performers first)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


org_code,org_name,parent_org,total_attendances,total_breaches,total_admissions,breach_rate_pct,performance_rank,performance_band,_gold_timestamp
RJN,EAST CHESHIRE NHS TRUST,NHS ENGLAND NORTH WEST,53343,28543,11881,53.51,1,RED,2026-06-30T13:00:42.310Z
RXW,THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST,NHS ENGLAND MIDLANDS,161144,75554,67276,46.89,2,RED,2026-06-30T13:00:42.310Z
RD1,ROYAL UNITED HOSPITALS BATH NHS FOUNDATION TRUST,NHS ENGLAND SOUTH WEST,107665,45075,58004,41.87,3,RED,2026-06-30T13:00:42.310Z
RTP,SURREY AND SUSSEX HEALTHCARE NHS TRUST,NHS ENGLAND SOUTH EAST,122739,50426,48863,41.08,4,RED,2026-06-30T13:00:42.310Z
RBL,WIRRAL UNIVERSITY TEACHING HOSPITAL NHS FOUNDATION TRUST,NHS ENGLAND NORTH WEST,130474,52703,47463,40.39,5,RED,2026-06-30T13:00:42.310Z
RWA,HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST,NHS ENGLAND NORTH EAST AND YORKSHIRE,170085,68113,71974,40.05,6,RED,2026-06-30T13:00:42.310Z
RX1,NOTTINGHAM UNIVERSITY HOSPITALS NHS TRUST,NHS ENGLAND MIDLANDS,252159,98503,107327,39.06,7,RED,2026-06-30T13:00:42.310Z
RBT,MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST,NHS ENGLAND NORTH WEST,120522,46910,36278,38.92,8,RED,2026-06-30T13:00:42.310Z
RMP,TAMESIDE AND GLOSSOP INTEGRATED CARE NHS FOUNDATION TRUST,NHS ENGLAND NORTH WEST,133155,50881,23081,38.21,9,RED,2026-06-30T13:00:42.310Z
RJR,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,NHS ENGLAND NORTH WEST,93701,35346,37059,37.72,10,RED,2026-06-30T13:00:42.310Z


In [0]:
# ============================================================
# CELL 5: Write Gold Tables
# ============================================================

# Create gold volume first
spark.sql("USE CATALOG healthcare_pipeline")
spark.sql("USE DATABASE ae_data")
spark.sql("CREATE VOLUME IF NOT EXISTS gold_volume")

gold_tables = {
    "gold_monthly_summary": gold_monthly_summary,
    "gold_hospital_ranking": gold_hospital_ranking,
}

for table_name, df in gold_tables.items():
    path = f"/Volumes/healthcare_pipeline/ae_data/gold_volume/delta/{table_name}"
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(path)
    
    print(f"Written: {table_name} ({df.count()} records)")

print()
print("All Gold tables written successfully")

Written: gold_monthly_summary (12 records)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Written: gold_hospital_ranking (198 records)

All Gold tables written successfully
